# Alias-config reconstruction — PRRSV

Đưa tool một **PRRSV mới tinh** (chỉ tên gene của reference, chưa có alias map): tầng gợi ý dựa-trên-tọa-độ
dựng lại được config curated tới đâu?

**Ba bước, gọi ĐÚNG hàm pipeline** (`ui/stages/bootstrap_alias.py`), không reimplement:
`build_coordinate_supported_alias_suggestions` → `review_uncertain_alias_suggestions` (LLM) →
`apply_approved_alias_suggestions` (dựng `config_temp`).

> **Đo hai chiều, per-canonical:**
> - **PRECISION** (duyệt config_temp): tool bỏ tên gì dưới canon X → truth có đồng ý thuộc X không?
>   Bắt `wrong_gene` (map sai), `false_save` (lưu nhầm rác).
> - **RECALL** (duyệt config_truth, **gate theo corpus**): alias thật truth có dưới X **mà thực sự xuất
>   hiện trong 100 record** → config_temp có giữ ở X không? Bắt `missed` (kiểu bug sM: alias thật bị loại).
>
> **Gate corpus (quan trọng):** alias truth **không** xuất hiện trong 100 record thì tool không có gì để
> gợi ý → **park, không tính accuracy**. Nếu không gate sẽ phạt oan tool.
>
> **Bước người không tự động được:** harness dừng ở khuyến nghị + chính sách auto-approve khai báo rõ
> (chấp nhận mọi dòng tool đánh `save`). Số là **cận trên**. Không API key → LLM mock (in rõ chế độ).

## Setup

In [1]:
from pathlib import Path
import os, sys
import pandas as pd

ROOT = Path.cwd()
for c in [ROOT, *ROOT.parents]:
    if (c / "app" / "src").exists():
        ROOT = c; break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "app" / "validation" / "03_alias_suggestion"))

# Nạp .env (giống UI) để LLMConfig thấy OPENAI_API_KEY
env = ROOT / ".env"
if env.exists():
    for line in env.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("="); os.environ.setdefault(k.strip(), v.strip())

import importlib
import config_reconstruction as R
importlib.reload(R)
from app.src.llm.config import LLMConfig

VIRUS = "PRRSV"
MIN_IOU = 0.90
PROVIDER = None if LLMConfig.from_env().available else R._MockLLMProvider()
print("LLM:", "REAL" if PROVIDER is None else "MOCK (no API key)")

LLM: REAL


## Chạy — seed → suggest → LLM → config_temp → so per-canonical

Ghi ra `outputs/config_temp_PRRSV.json`, `reconstruction_PRRSV.tsv` (per-canonical), `..._detail.tsv` (từng ca sai).

In [2]:
per_canon, detail = R.run_virus(VIRUS, R.DATASETS[VIRUS], MIN_IOU, llm_provider=PROVIDER)
R.summarize(VIRUS, per_canon)


===== PRRSV =====
  PRECISION (config_temp → truth): 64/129 = 49.6%
     wrong_gene 1  false_save 1  not_in_truth 63
  RECALL (truth ∩ corpus → config_temp): 62/122 = 50.8%
     missed 60  (parked, not scored: 44)


## Bảng per-canonical (đọc từng gene)

In [3]:
pd.set_option("display.width", 200)
per_canon

,canonical,temp_saved,precision_correct,wrong_gene,false_save,not_in_truth,precision_pct,recall_denom,recall_found,missed,recall_pct,parked
0,ORF1a,20,8,1,0,11,40.0,15,7,8,46.7,5
1,ORF1ab,0,0,0,0,0,NaN,0,0,0,NaN,7
2,ORF1b,20,9,0,1,10,45.0,18,8,10,44.4,3
3,ORF2a,7,4,0,0,3,57.1,16,4,12,25.0,6
4,ORF2b,12,6,0,0,6,50.0,15,6,9,40.0,3
5,ORF3,16,10,0,0,6,62.5,11,10,1,90.9,4
6,ORF4,19,11,0,0,8,57.9,12,11,1,91.7,3
7,ORF5,21,11,0,0,10,52.4,13,11,2,84.6,4
8,ORF5a,0,0,0,0,0,NaN,0,0,0,NaN,5
9,ORF6,8,3,0,0,5,37.5,14,3,11,21.4,2


## Precision — chỗ tool lưu SAI

`wrong_gene` (map nhầm gene) và `false_save` (lưu nhầm rác) là lỗi đắt, lý tưởng = 0.
`not_in_truth` = tool gợi ý mà truth im lặng → **kiểm truth có thiếu không**.

In [4]:
prec_issues = detail[detail["side"]=="precision"]
prec_issues if len(prec_issues) else "— precision sạch, không lưu sai gì —"

,canonical,side,name,issue
0,ORF1a,precision,orf1ab,wrong_gene(ORF1ab)
1,ORF1a,precision,orf1apolyprotein;polypeptideprecursorofnonstru...,not_in_truth
2,ORF1a,precision,orf1a;papainlikecysteineproteasealphaandbeta;c...,not_in_truth
3,ORF1a,precision,orf1a;papainlikecysteineproteasealphaandbeta;c...,not_in_truth
4,ORF1a,precision,orf1a;papainlikecysteineproteasealphaandbeta;c...,not_in_truth
...,...,...,...,...
143,ORF6,precision,nonglycosylate;orf6,not_in_truth
157,ORF7,precision,nucleocapsidprotein;orf7,not_in_truth
158,ORF7,precision,structuralprotein;n;orf7,not_in_truth
159,ORF7,precision,n;orf7,not_in_truth


## Recall — chỗ tool BỎ SÓT (đây là chỗ bắt bug kiểu sM)

`missed(temp→...)` cho biết alias thật bị đẩy đi đâu: `excluded` = bị loại nhầm (nguy hiểm nhất),
một canon khác = map lệch.

In [5]:
miss = detail[(detail["side"]=="recall") & (detail["issue"].str.startswith("missed"))]
miss if len(miss) else "— recall đầy đủ, không bỏ sót alias nào có trong corpus —"

,canonical,side,name,issue
13,ORF1a,recall,papainlikecysteineprotease,missed(temp→excluded)
17,ORF1a,recall,1areplicaseprotein;papainlikecysteine/serinepr...,missed(temp→excluded)
19,ORF1a,recall,nonstructuralpolyprotein1a;encodespapainlikecy...,missed(temp→excluded)
20,ORF1a,recall,containspapainlikecysteineprotease,missed(temp→excluded)
21,ORF1a,recall,containspapainlikecysteineproteinasedomains1a(...,missed(temp→None)
22,ORF1a,recall,containscysteineproteinase2domain(cp2)thatmedi...,missed(temp→None)
23,ORF1a,recall,"encodespapainlikecycteineprotease,3clikecystei...",missed(temp→excluded)
24,ORF1a,recall,polypeptideprecursorofnonstructuralproteinsnsp...,missed(temp→excluded)
44,ORF1b,recall,rnadependentrnapolymerase,missed(temp→excluded)
45,ORF1b,recall,rnadependentrnapolymerase/helicase,missed(temp→excluded)


## Parked — alias truth KHÔNG có trong corpus (không tính điểm)

Đây là alias curated từ literature/kinh nghiệm mà lô 100 record này không dùng. Tool không thể gợi ý được,
nên loại khỏi mẫu số recall. Liệt kê để minh bạch.

In [6]:
parked = detail[detail["issue"]=="parked_not_in_corpus"]
print(f"{len(parked)} alias park (không tính accuracy)")
parked[["canonical","name"]]

32 alias park (không tính accuracy)


,canonical,name
12,ORF1a,polyprotein1a
14,ORF1a,polyproteinpp1a
15,ORF1a,pp1a
16,ORF1a,1areplicaseprotein
18,ORF1a,nonstructuralpolyprotein1a
43,ORF1b,polyprotein1b
47,ORF1b,pp1b
51,ORF1b,nonstructualpolyprotein1b
60,ORF2a,glycosylatedprotein2a
61,ORF2a,gp2aprotein


## Ghi chú cho paper

- **Precision** = độ tin của cái tool tự lưu; **Recall** (gate corpus) = độ phủ trên phần tool *có thể*
  gợi ý. Hai số tách riêng, không gộp.
- **Gate corpus** là bắt buộc để công bằng: alias literature-only không phải lỗi lifting.
- **Recall detail là nơi lộ bug kiểu sM** — alias thật có trong corpus mà bị `missed(temp→excluded)`.